# ELT — Leitos Hospitalares

**Diferença em relação ao ETL:**
- No **ETL**, os dados são transformados em Python antes de entrar no banco.
- No **ELT**, os dados brutos entram no banco primeiro (schema `raw`), e todas as transformações acontecem dentro do PostgreSQL via **Views SQL** (schema `elt`).

**Fluxo:**
```
CSVs → Python (leitura mínima) → raw.leitos_bruto → Views SQL → elt.dim_* + elt.fato_*
```

## 1. Instalação e imports

In [ ]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import os
import csv
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Carrega variáveis do arquivo .env
load_dotenv()

True

## 2. Configuração da conexão com PostgreSQL

In [ ]:
USUARIO  = os.getenv("DB_USER", "postgres")
SENHA    = os.getenv("DB_PASSWORD", "senha_secreta")
HOST     = os.getenv("DB_HOST", "localhost")
PORTA    = os.getenv("DB_PORT", "5432")
BANCO    = os.getenv("DB_NAME", "projeto")

DATABASE_URL = f"postgresql://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO}"
engine = create_engine(DATABASE_URL)

print(f"Conexão criada com sucesso: {BANCO}@{HOST}:{PORTA}")

Conexão criada com sucesso: projeto@localhost:5432


## 3. EXTRACT — Leitura mínima dos CSVs

No ELT, o Python faz o mínimo necessário:
- Detectar separador
- Ler com encoding correto
- Adicionar coluna de rastreabilidade (`ano_origem`)
- **Sem limpeza, sem transformação**

In [ ]:
ARQUIVOS_LEITOS = {
    2023: "../database/data_raw/Leitos_2023.csv",
    2024: "../database/data_raw/Leitos_2024.csv",
    2025: "../database/data_raw/Leitos_2025.csv"
}

# Colunas que devem ser lidas como texto (para preservar zeros à esquerda)
COLUNAS_CODIGO = {
    "COMP": "string",
    "CO_IBGE": "string",
    "CNES": "string",
    "CO_TIPO_UNIDADE": "string",
    "NATUREZA_JURIDICA": "string",
    "CO_CEP": "string",
    "NU_ENDERECO": "string"
}

In [ ]:
def detectar_encoding(caminho_arquivo):
    """Detecta o encoding correto do arquivo."""
    encodings = ["latin-1", "utf-8", "iso-8859-1", "cp1252"]
    
    for encoding in encodings:
        try:
            with open(caminho_arquivo, "r", encoding=encoding, newline="") as f:
                f.read(4096)
            return encoding
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    return "latin-1"  # fallback

def detectar_separador(caminho_arquivo):
    """Detecta automaticamente se o CSV usa vírgula ou ponto e vírgula."""
    with open(caminho_arquivo, "r", encoding=detectar_encoding(caminho_arquivo), newline="") as arquivo:
        amostra = arquivo.read(4096)
    return csv.Sniffer().sniff(amostra, delimiters=",;").delimiter


def extrair_csv(caminho_arquivo, ano_origem):
    """
    Extrai um CSV com o mínimo de intervenção.
    """
    encoding = detectar_encoding(caminho_arquivo)
    print(f" {os.path.basename(caminho_arquivo)} - Encoding: {encoding}")

    separador = detectar_separador(caminho_arquivo)
    print(f"     Separador: '{separador}'")

    df = pd.read_csv(
        caminho_arquivo,
        sep=separador,
        encoding="latin-1",
        dtype=COLUNAS_CODIGO,
        low_memory=False
    )

    df.columns = df.columns.str.strip().str.lower()

    df["arquivo_origem"] = os.path.basename(caminho_arquivo)
    df["ano_origem"]     = str(ano_origem)

    return df

In [ ]:
bases = []
for ano, caminho in ARQUIVOS_LEITOS.items():
    try:
        df_ano = extrair_csv(caminho, ano)
        print(f" {ano}: {df_ano.shape[0]:,} linhas, {df_ano.shape[1]} colunas\n")
        bases.append(df_ano)
    except FileNotFoundError:
        print(f" Arquivo não encontrado: {caminho}\n")
    except Exception as e:
        print(f" Erro ao ler {caminho}: {e}\n")

if not bases:
    raise ValueError("Nenhum arquivo CSV foi carregado com sucesso!")

df_raw = pd.concat(bases, ignore_index=True, sort=False)

print(f"\nBase unificada: {df_raw.shape[0]:,} linhas, {df_raw.shape[1]} colunas")